In [1]:
import pandas as pd
from bs4 import BeautifulSoup
import requests
import re
from tqdm import tnrange
from tqdm import tqdm
import nltk
from nltk.corpus import stopwords
import pickle as pickle
from konlpy.tag import Twitter
from collections import Counter
import konlpy

In [ ]:
content_list = pd.read_pickle("content_list.pkl") # 11 개 기업 자소서 내용
counter_token = pd.read_pickle("counter_2.pkl") # 11개 기업의 자소서 내용을 불처리하고 토큰화한것 딕트
essay_df = pd.read_pickle("essay_df_2.pkl") #대분류 소분류도 긁어온 크롤링 전체 자료
new_del_word = pd.read_pickle("new_del_word.pkl") # set 불용어 추가한 것들 기업이름은 현대만 들어가있음
position_counter_list = pd.read_pickle("position_counter_list.pkl") # 14개 대분류 직무별 자소서 내용 불처리하고 토근화한것
sort_counter_list = pd.read_pickle("sorted_counter_2.pkl") #counter_2 정렬 튜플형식 11개 기업
stopwords = pd.read_json('stopwords-ko.json', encoding = "UTF-8")
my_del_word = pd.read_pickle('my_del_word.pkl') #재실행용
test_df = pd.read_pickle('position_essay_n_nan.pkl') #넌 값 아닌 직무기업정렬
pos_essay_df = pd.read_pickle('pos_essay_df.pkl') #position_essay_n_nan에서 자소서 pos작업하고 str로 각자 묶은것
company_affiliate_df = pd.read_pickle("company_affiliate.pkl") #계열사와 기업그룹이름 짝짓기
company_code_dict = pd.read_pickle("company_code_dict.pkl")

In [ ]:
position_essay_all = essay_df.sort_values(["position_broad","company_id"])#직무,기업 순 정렬
position_essay_n_nan = position_essay_all[position_essay_all["feedback"].notnull() & position_essay_all["score"].notnull()]
 #position_essay_all에서 넌 값 아닌 데이터 프레임 만들기

In [ ]:
position_essay_n_nan.reset_index(inplace=True, drop=True)
test_df = position_essay_n_nan

#### 미리 형태소 & 불용어처리

In [ ]:
my_del_word = list(my_del_word)
stopwords = list(stopwords)
common_del_list = ['되어다','때문','그리고','좋다','자다','없다','같다','싶다','보다', '하다', '있다', '되다', '이다', '돼다', '않다', '그렇다', '아니다', '이렇다', '그렇다', '어떻다']
del_list = stopwords + common_del_list + my_del_word
counter_list = []
def not_counter(contents): 
    
    twitter = Twitter()
    raw_pos_tagged = twitter.pos(contents, norm=True, stem=True) 
 
    word_cleaned = []
    
    for word, tag in raw_pos_tagged: 
        if not tag in ["Josa", "Eomi", "Punctuation", "Foreign"]: 
            if (len(word) != 1) & (word not in del_list): 
                word_cleaned.append(word)

    return word_cleaned

In [ ]:
for x in position_essay_n_nan.index:
    content = position_essay_n_nan["content"][x]
    position_essay_n_nan["content"] = ' '.join(not_counter(content)

In [ ]:
position_essay_all.to_pickle('position_essay_all.pkl')
position_essay_n_nan.to_pickle('position_essay_n_nan.pkl')
test_df.to_pickle('pos_essay_df.pkl')

# 데이터프레임 참고용
http://blog.naver.com/PostView.nhn?blogId=rising_n_falling&logNo=221622971970&categoryNo=15&parentCategoryNo=12&viewDate=&currentPage=&postListTopCurrentPage=&isAfterWrite=true